<a href="https://colab.research.google.com/github/EricChen356/video-background-remover/blob/main/SAM3_Video_Background_Remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM 3 Video Background Remover  ·  green-screen / alpha / transparent

SAM 3 -> MatAnyone2 pipeline


Keeps the **foreground subject(s)** and replaces the background with a green screen,
holding up on **fine hair edges** and staying **temporally stable** across scene cuts.

**Pipeline (per video):**
1. **Probe** metadata (fps, frames, resolution, audio).
2. **Scene-cut detection** with **PySceneDetect** (`AdaptiveDetector` + `ThresholdDetector` for fades).
3. **Per shot** (fresh SAM3 session so tracking memory never smears across a cut):
   - **SAM 3** (HF `transformers`, *text* prompt `"person"`) → per-frame instance masks + stable `object_id`s.
   - **Foreground selection** via **Depth Anything V2** (median depth inside each mask) + geometry tie-breakers.
   - **MatAnyone** mask-guided matting, seeded from the coarse mask → soft alpha with hair detail.
     Re-seeded when a new subject becomes foreground mid-shot.
4. **Composite** in float: `out = fg*alpha + green*(1-alpha)`.
5. **Reassemble** at original fps, **mux audio**, write atomically, resume-skip.

> Verified against the docs before coding: SAM3 video uses **text prompts only** (no boxes);
> `facebook/sam3.1` is weights-only with no `transformers` integration, so the transformers path
> uses `facebook/sam3`. Depth Anything V2 outputs **relative inverse depth** (larger = nearer).

In [ ]:
#  ──── INSTALL DEPENDENCIES ──────────────────────────────────────────────────────────────────────
import os
from google.colab import userdata
!git clone https://github.com/pq-yang/MatAnyone2.git
%cd MatAnyone2
!pip install -e . -q

!pip install git+https://github.com/facebookresearch/sam3.git -q

!pip install ultralytics -q

!pip install torch torchvision -q

!pip install "imageio>=2.33,!=2.35.0" "huggingface-hub>=1.5.0,<2.0" -q

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print('\n✅ All dependencies installed.')


In [ ]:
#  ──── INSTALL MODEL WEIGHTS ────────────────────────────────────────────────
import os, requests, tqdm

MA2_PATH = 'pretrained_models/matanyone2.pth'
MA2_URL  = 'https://github.com/pq-yang/MatAnyone2/releases/download/v1.0.0/matanyone2.pth'

if os.path.exists(MA2_PATH):
    os.remove(MA2_PATH)
    print('Removed corrupt checkpoint')

os.makedirs('pretrained_models', exist_ok=True)

response = requests.get(MA2_URL, stream=True)
total = int(response.headers.get('content-length', 0))

with open(MA2_PATH, 'wb') as f, tqdm.tqdm(
    desc='matanyone2.pth',
    total=total,
    unit='B',
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        bar.update(len(chunk))

print(f'Downloaded: {os.path.getsize(MA2_PATH) / 1e6:.1f} MB')

In [ ]:
# ──── USER CONFIG ────────────────────────────────────────────────────────────────────────
# Everything you normally tune lives here. Defaults are conservative and work well
# for typical single/multi-person footage — change values, not the cells below.
import os as _os
import numpy as np

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_DIR   = '/content/videos'           # folder of input videos (processed as a batch)
OUTPUT_DIR  = '/content/matanyone2_out'   # mattes / alpha / debug masks are written here

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_MODE = 'greenscreen'   # 'greenscreen' | 'alpha' | 'transparent'
GREEN       = np.array([120, 255, 152], dtype=np.float32) / 255.0   # key colour for greenscreen

# ── Performance ───────────────────────────────────────────────────────────────
MAX_SIZE    = 0               # 0 = full resolution; else matte at this short-side size

# -- Encoding -----------------------------------------------------------------
WRITE_ALPHA_VIDEO = True
ENCODER_PRESET    = 'veryfast'
ENCODER_CRF       = 18

# ── Matting quality (MatAnyone 2) ─────────────────────────────────────────────
N_WARMUP_STATIC = 20
N_WARMUP_MOTION = 20
R_ERODE     = 0
R_DILATE    = 20      # seed mask is dilated this many px before matting — generous so
                      # thin held items (mic stalks, cables) fall inside the refine band.

# ── Detection thresholds ──────────────────────────────────────────────────────
YOLO_CONF     = 0.3   # detection confidence floor (people AND held props)
CUT_THRESHOLD = 0.15  # scene-cut sensitivity (lower = more cuts)

# BOX_SCORE_RATIO: YOLO person boxes are scored by size + centrality + confidence.
#   Only boxes scoring >= this fraction of the best box are passed to SAM 3.
#   Lower this (e.g. 0.1) if YOLO detects a person who is not being masked.
BOX_SCORE_RATIO = 0.3

# SAM_MASK_THRESHOLD: threshold on the CONCEPT head's mask logits — the clean,
#   high-quality subject silhouette. 0.5 is the original tuned value; lower toward
#   0.0 for a slightly more inclusive subject edge, raise for a tighter one.
SAM_MASK_THRESHOLD = 0.5

# ── Concept-head reliability (why SAM 3 may not segment a person YOLO found) ──
# SAM 3's concept head is a CONFIDENCE-GATED detector — it only returns a mask when it
# is confident the region holds the concept, so a hard subject (back turned, low
# contrast, occluded) can be dropped entirely even though YOLO detected it. These knobs
# make it fire reliably so a clean silhouette exists to gate the interactive head.
#
# SAM_DETECT_THRESHOLD: concept-detector confidence gate (sigmoid prob). Lower it so
#   borderline subjects are still segmented; None = leave the model default. Raise it if
#   spurious extra detections start appearing.
SAM_DETECT_THRESHOLD = 0.2
# CONCEPT_TEXT_DETECT: also run an exhaustive text ("person") detection and union the
#   instance matching each YOLO box (by IoU) into the concept silhouette. Text detection
#   is SAM 3's best-tuned path and recovers people the bare box-geometric prompt misses.
CONCEPT_TEXT_DETECT = True
CONCEPT_TEXT_PROMPT = 'person'
CONCEPT_MATCH_IOU   = 0.3    # min YOLO-box vs concept-instance IoU to treat as same subject

# FORCE_INTERACTIVE: also run SAM 3's forceful interactive box->mask head and union it
#   with the concept mask. It has NO concept gate, so it always fires — recovering held
#   objects AND a subject the concept head missed — but it also fills in background the
#   subject wraps around. INTERACTIVE_GATE_BG (below) strips that fill back out.
#   Set False for concept-only behaviour (held objects / missed subjects may be lost).
FORCE_INTERACTIVE = True

# INTERACTIVE_MASK_THRESHOLD: threshold on the interactive head's logits (0.0 is its
#   natural boundary). Lower to pull in more of a held object; raise to be stricter.
INTERACTIVE_MASK_THRESHOLD = 0.0

# BOX_CLIP_MARGIN: the interactive box->mask head can occasionally leak onto a
#   high-contrast region far from the prompt box. We keep only mask blobs that
#   overlap the box expanded by this fraction on every side, which drops far-field
#   leaks while keeping the WHOLE subject blob (it is never cut at the box edge).
#   Raise it if a held item sits just outside the detected person box.
BOX_CLIP_MARGIN = 0.10

# INTERACTIVE_GATE_BG / GATE_BG_MARGIN: the interactive head both grabs held props AND
#   fills in background the body wraps around (e.g. the arm-on-hip <-> torso gap) — and
#   topology can't tell them apart (both are pockets the body surrounds). This gate uses
#   APPEARANCE: a region the interactive head adds is dropped only when its colour is
#   clearly more like the scene background than like the subject, so background showing
#   through the body is removed while a held prop (even one YOLO never detected, like a
#   lav mic) is kept. YOLO prop points are always kept. GATE_BG_MARGIN (0..1) is how
#   decisively background-like a region must be to drop — higher keeps more (safer for
#   props), lower removes more background. Set INTERACTIVE_GATE_BG = False to union all.
INTERACTIVE_GATE_BG = True
GATE_BG_MARGIN = 0.05

# RECOVER_HELD_OBJECTS: SAM's interactive head can miss a held object it doesn't
#   recognise (e.g. a lav-mic transmitter), and YOLO may never detect it, so it ends up
#   keyed out. This adds back regions INSIDE a person box that the masks missed when they
#   (a) are surrounded by the subject (held against the body) AND (b) do NOT look like the
#   scene background — i.e. a distinct object the person is holding. Background showing
#   through the body and regions the subject doesn't surround are left out. (Inverse of
#   the bg gate; shares its appearance model.) Set False to disable.
RECOVER_HELD_OBJECTS  = True
RECOVER_SURROUND_FRAC = 0.6   # min fraction of a region's border that must be the subject
RECOVER_BG_MAXPROB    = 96    # 0..255; regions more background-like than this are NOT recovered

# ── Held-object handling ──────────────────────────────────────────────────────
# The interactive box→mask already grabs items held against the body. For props
# YOLO can also recognise, we additionally drop a positive SAM 3 point on them so
# they are pulled into the subject's mask instead of being keyed out as background.
MASK_HELD_OBJECTS    = True
HELD_OBJECT_MAX_FRAC = 0.25   # ignore "props" larger than this fraction of frame area
                              # (avoids grabbing background furniture inside the box)
# COCO class ids for small objects a person is likely to hold. (A microphone is not a
# COCO class, but hand-held recorders/mics often register as cell phone / remote.)
HELD_OBJECT_CLASSES = {
    24, 25, 26, 28,        # backpack, umbrella, handbag, suitcase
    32,                    # sports ball
    39, 40, 41,            # bottle, wine glass, cup
    43, 44,                # knife, spoon
    63, 64, 65, 66, 67,    # laptop, mouse, remote, keyboard, cell phone
    73, 74, 76, 77, 79,    # book, clock, scissors, teddy bear, toothbrush
}

# FILL_MASK_HOLES / MAX_HOLE_FRAC: flood-fill ONLY small enclosed holes (speckle, minor
#   mask gaps) up to MAX_HOLE_FRAC of the subject area. A large genuine negative-space
#   region — e.g. the gap between an arm-on-hip and the torso — is LEFT OPEN so the
#   background shows through. Set MAX_HOLE_FRAC = 1.0 to fill every hole, or
#   FILL_MASK_HOLES = False to never fill.
FILL_MASK_HOLES = True
MAX_HOLE_FRAC   = 0.01

# ── Troubleshooting ───────────────────────────────────────────────────────────
SAVE_SEED_MASKS = True

VIDEO_EXTENSIONS = ('*.mp4', '*.mov', '*.avi', '*.mkv')

_os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded.')


In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# TF32 matmuls on Ampere+ GPUs — a near-free speedup with no visible quality change.
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ── MatAnyone 2 (video matting) ───────────────────────────────────────────────
# Loaded and kept in float32 — do NOT wrap in global bfloat16 autocast, as that
# degrades matting quality for borderline pixels (fine hair, held objects, etc.).
from hydra.core.global_hydra import GlobalHydra
from matanyone2.utils.get_default_model import get_matanyone2_model

GlobalHydra.instance().clear()
matanyone2_model = get_matanyone2_model(MA2_PATH, device)
matanyone2_model = matanyone2_model.float()
print('✅ MatAnyone 2 loaded')

# ── SAM 3 (box / text → pixel-precise mask) ───────────────────────────────────
# Built with enable_inst_interactivity=True so the SAM 1-style interactive predictor
# (sam3_model.inst_interactive_predictor) is loaded from the same facebook/sam3
# checkpoint. That predictor takes a box and segments whatever is INSIDE it —
# straight to the segmentation head, with NO detector / concept re-analysis and NO
# confidence gate — which is what keeps hand-held objects (mics, phones) in the
# foreground instead of being keyed green. Sam3Processor's CONCEPT head makes
# the clean subject silhouette; the interactive head is UNIONED in to force-
# segment held objects the concept head drops. SAM 3 weights are bfloat16;
# autocast is applied
# locally around each SAM 3 call so MatAnyone2 (float32) is unaffected.
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

sam3_model = build_sam3_image_model(device=device, eval_mode=True,
                                    enable_inst_interactivity=True)
sam3_predictor   = Sam3Processor(sam3_model, device=device)   # text-prompt fallback
sam3_interactive = sam3_model.inst_interactive_predictor      # box→mask head
# NOTE: call it via sam3_model.predict_inst(state, box=...) — the predictor
# reuses the main backbone features cached by Sam3Processor.set_image(); it has
# no backbone of its own, so inst_interactive_predictor.set_image() would fail.
assert sam3_interactive is not None, \
    'inst_interactive_predictor is None — rebuild with enable_inst_interactivity=True'
print('✅ SAM 3 loaded (interactive box→mask enabled)')

# ── YOLOv8n (person + held-prop detector → seed boxes/points for SAM 3) ────────
from ultralytics import YOLO
yolo_model = YOLO('yolov8n.pt')
print('✅ YOLOv8n loaded')


In [ ]:
import time
import torch.nn.functional as F
import imageio
from tqdm import tqdm
from contextlib import contextmanager, nullcontext
from collections import OrderedDict
from matanyone2.inference.inference_core import InferenceCore
from matanyone2.utils.inference_utils import gen_dilate, gen_erosion

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 0 — STEP TIMER
# ─────────────────────────────────────────────────────────────────────────────
class StepTimer:
    ORDER = ['decode', 'scene_cut', 'yolo', 'sam', 'mask_debug',
             'matting_pre', 'matting_warmup', 'matting_infer', 'matting_post',
             'composite', 'encode']

    def __init__(self, sync_cuda: bool = True):
        self.totals = OrderedDict()
        self.counts = OrderedDict()
        self.sync_cuda = bool(sync_cuda) and torch.cuda.is_available()

    @contextmanager
    def track(self, name: str):
        if self.sync_cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        try:
            yield
        finally:
            if self.sync_cuda:
                torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            self.totals[name] = self.totals.get(name, 0.0) + dt
            self.counts[name] = self.counts.get(name, 0) + 1

    def merge(self, other: 'StepTimer'):
        for k, v in other.totals.items():
            self.totals[k] = self.totals.get(k, 0.0) + v
            self.counts[k] = self.counts.get(k, 0) + other.counts[k]

    def report(self, title: str = 'Timing breakdown', n_frames: int | None = None):
        keys  = [k for k in self.ORDER if k in self.totals]
        keys += [k for k in self.totals if k not in self.ORDER]
        total = sum(self.totals.values())
        print(f'\n{"─" * 70}')
        print(f'⏱  {title}' + (f'   ({n_frames} output frames)' if n_frames else ''))
        print(f'{"─" * 70}')
        print(f'   {"step":<16}{"time":>10}{"share":>9}{"calls":>8}{"ms/call":>11}')
        for k in keys:
            t, c = self.totals[k], self.counts[k]
            pct  = 100 * t / total if total else 0.0
            print(f'   {k:<16}{t:>9.2f}s{pct:>8.1f}%{c:>8}{1000 * t / c:>10.1f}')
        print(f'   {"·" * 60}')
        line = f'   {"TOTAL":<16}{total:>9.2f}s{100.0:>8.1f}%'
        if n_frames:
            line += f'{n_frames:>8}{1000 * total / n_frames:>10.1f}  (/out frame)'
        print(line)
        print(f'{"─" * 70}')


def _track(timer: 'StepTimer | None', name: str):
    return timer.track(name) if timer is not None else nullcontext()

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 2 — SCENE-CUT DETECTION
# ─────────────────────────────────────────────────────────────────────────────
def compute_scene_cuts(frames_bgr: list[np.ndarray],
                       threshold: float = 0.30) -> tuple[list[int], list[float]]:
    boundaries = [0]
    scores: list[float] = []
    prev_hist = None
    for idx, frame in enumerate(frames_bgr):
        hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = [cv2.calcHist([hsv], [c], None, [64], [0, 256]) for c in range(3)]
        for h in hist:
            cv2.normalize(h, h)
        if prev_hist is not None:
            diffs = [cv2.compareHist(prev_hist[c], hist[c], cv2.HISTCMP_BHATTACHARYYA)
                     for c in range(3)]
            score = float(np.mean(diffs))
            scores.append(score)
            if score > threshold:
                boundaries.append(idx)
        prev_hist = hist
    return boundaries, scores

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 3 — AUTO FOREGROUND MASK
# ─────────────────────────────────────────────────────────────────────────────
def _enclosed_holes(mask_u8: np.ndarray) -> np.ndarray:
    """Binary (0/255) map of background pixels FULLY ENCLOSED by foreground.

    Pads with a 1-px background border so every background region touching the image
    edge is reached by the flood seed; whatever the flood cannot reach is background the
    foreground completely surrounds (e.g. the gap between an arm-on-hip and the torso).
    Flooding from a single corner instead would wrongly catch any background the
    foreground happens to cut off from that corner."""
    if not mask_u8.any():
        return np.zeros_like(mask_u8)
    h, w = mask_u8.shape
    padded = cv2.copyMakeBorder(mask_u8, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0)
    ff = padded.copy()
    flood = np.zeros((h + 4, w + 4), np.uint8)
    cv2.floodFill(ff, flood, (0, 0), 255)      # reach all edge-connected background
    return cv2.bitwise_not(ff)[1:-1, 1:-1]     # crop pad -> enclosed holes only


def _fill_holes(mask_u8: np.ndarray, max_hole_frac: float = 1.0) -> np.ndarray:
    """Fill fully-enclosed holes in a binary (0/255) mask, up to a size limit.

    Of the enclosed holes (see _enclosed_holes), ONLY the ones whose area is
    <= max_hole_frac of the subject mask area are filled — so speckle and minor mask
    gaps are sealed, but a large genuine negative-space region (e.g. the gap between an
    arm-on-hip and the torso) is LEFT OPEN. max_hole_frac >= 1.0 fills every hole.
    """
    if not mask_u8.any():
        return mask_u8
    holes = _enclosed_holes(mask_u8)
    if max_hole_frac >= 1.0:
        return cv2.bitwise_or(mask_u8, holes)   # fill everything (original behaviour)
    max_area = max(1, int(max_hole_frac * int((mask_u8 > 0).sum())))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(holes, connectivity=8)
    small = np.zeros_like(holes)
    for lbl in range(1, n):                     # 0 = the non-hole background label
        if stats[lbl, cv2.CC_STAT_AREA] <= max_area:
            small[labels == lbl] = 255
    return cv2.bitwise_or(mask_u8, small)


def _keep_components_in_box(mask_u8: np.ndarray, x1, y1, x2, y2,
                            margin: float = 0.0) -> np.ndarray:
    """Keep only connected components that touch the (margin-expanded) box.

    Removes far-field segmentation leaks while keeping every blob belonging to the
    prompted subject. WHOLE components are kept, so the subject is never cut along
    the box edge (unlike a hard rectangular clip)."""
    if not mask_u8.any():
        return mask_u8
    h, w = mask_u8.shape
    mx, my = (x2 - x1) * margin, (y2 - y1) * margin
    bx1, by1 = max(0, int(x1 - mx)), max(0, int(y1 - my))
    bx2, by2 = min(w, int(x2 + mx)), min(h, int(y2 + my))
    n, labels = cv2.connectedComponents(mask_u8, connectivity=8)
    keep_labels = set(np.unique(labels[by1:by2, bx1:bx2]).tolist()) - {0}
    if not keep_labels:
        return np.zeros_like(mask_u8)
    return np.where(np.isin(labels, list(keep_labels)), 255, 0).astype(np.uint8)


def _drop_background_regions(add_mask: np.ndarray, ref_mask: np.ndarray,
                            frame_rgb: np.ndarray, prop_pts: list | None = None,
                            margin: float = 0.05, min_area: int = 64) -> np.ndarray:
    """Drop interactive-added regions whose APPEARANCE matches the scene background.

    A held prop and a background gap (e.g. the arm-on-hip <-> torso gap) are BOTH
    pockets the body surrounds, so topology can't tell them apart — but appearance can:
    a background gap shows the actual scene background, while a held prop looks like a
    distinct object. For every region the interactive head adds beyond the concept
    silhouette `ref_mask`, this compares how background-like vs subject-like its colours
    are (HSV histogram back-projection) and drops only the ones that are clearly more
    background-like — so background showing through the body is removed while a held prop
    (even one YOLO never detected) is kept.

    `margin` (0..1) is how decisively background-like a region must be to drop (higher =
    keep more, safer for props). Regions with a YOLO prop point are always kept."""
    if not add_mask.any() or not ref_mask.any():
        return add_mask
    extra = cv2.bitwise_and(add_mask, cv2.bitwise_not(ref_mask))
    if not extra.any():
        return add_mask

    hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
    subj_est = cv2.dilate(cv2.bitwise_or(add_mask, ref_mask),
                          cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25)))
    bg_sel = (subj_est == 0).astype(np.uint8)
    if int(bg_sel.sum()) < 100:                 # subject fills frame -> cannot model bg
        return add_mask
    subj_sel = (ref_mask > 0).astype(np.uint8)

    bins, ranges = [36, 50], [0, 180, 0, 256]   # HSV hue+saturation
    h_bg = cv2.calcHist([hsv], [0, 1], bg_sel, bins, ranges)
    h_subj = cv2.calcHist([hsv], [0, 1], subj_sel, bins, ranges)
    cv2.normalize(h_bg, h_bg, 0, 255, cv2.NORM_MINMAX)
    cv2.normalize(h_subj, h_subj, 0, 255, cv2.NORM_MINMAX)
    bp_bg = cv2.calcBackProject([hsv], [0, 1], h_bg, ranges, 1).astype(np.float32)
    bp_subj = cv2.calcBackProject([hsv], [0, 1], h_subj, ranges, 1).astype(np.float32)

    prop_pts = prop_pts or []
    n, labels, stats, _ = cv2.connectedComponentsWithStats(extra, connectivity=8)
    drop = np.zeros_like(add_mask)
    hh, ww = add_mask.shape
    for lbl in range(1, n):
        if stats[lbl, cv2.CC_STAT_AREA] < min_area:
            continue
        comp = labels == lbl
        if any(0 <= int(round(py)) < hh and 0 <= int(round(px)) < ww
               and comp[int(round(py)), int(round(px))] for (px, py) in prop_pts):
            continue                            # YOLO-confirmed prop -> always keep
        if float(bp_bg[comp].mean()) > float(bp_subj[comp].mean()) + margin * 255.0:
            drop[comp] = 255                    # clearly background-like -> remove
    return cv2.bitwise_and(add_mask, cv2.bitwise_not(drop))


def _recover_held_objects(subject_mask: np.ndarray, frame_rgb: np.ndarray, boxes: list,
                          surround_frac: float = 0.6, bg_maxprob: float = 96.0,
                          max_frac: float = 0.1, min_area: int = 64) -> np.ndarray:
    """Add held objects SAM missed: non-background regions the subject surrounds.

    SAM's interactive head can omit a held object it doesn't recognise (e.g. a lav-mic
    transmitter), and YOLO may never detect it, so it is keyed out. A held object, though,
    sits AGAINST the body: it is a region the subject wraps around AND it does NOT look
    like the scene background. This scans, inside each person box, regions the current
    mask missed and adds those that are (a) surrounded by the subject on at least
    `surround_frac` of their border and (b) less background-like than `bg_maxprob`
    (0..255 HSV back-projection). Background showing through the body (matches the scene
    background) and regions the subject does not surround (open background) are left out.
    Inverse of _drop_background_regions; shares the same appearance model.
    """
    if not subject_mask.any():
        return subject_mask
    h, w = subject_mask.shape
    hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
    subj_est = cv2.dilate(subject_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25)))
    bg_sel = (subj_est == 0).astype(np.uint8)
    if int(bg_sel.sum()) < 100:
        return subject_mask
    hbg = cv2.calcHist([hsv], [0, 1], bg_sel, [36, 50], [0, 180, 0, 256])
    cv2.normalize(hbg, hbg, 0, 255, cv2.NORM_MINMAX)
    bp_bg = cv2.calcBackProject([hsv], [0, 1], hbg, [0, 180, 0, 256], 1).astype(np.float32)
    subj_fg = subject_mask > 0
    k3 = np.ones((3, 3), np.uint8)
    add = np.zeros_like(subject_mask)
    for box in boxes:
        x1, y1, x2, y2 = (int(round(v)) for v in box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        roi = np.zeros_like(subject_mask)
        roi[y1:y2, x1:x2] = 255
        cand = cv2.bitwise_and(roi, cv2.bitwise_not(subject_mask))   # unmasked area in the box
        if not cand.any():
            continue
        n, lab, st, _cen = cv2.connectedComponentsWithStats(cand, connectivity=8)
        for L in range(1, n):
            a = int(st[L, cv2.CC_STAT_AREA])
            if a < min_area or a > max_frac * w * h:
                continue
            comp = lab == L
            cu = comp.astype(np.uint8)
            border = cv2.dilate(cu, k3) & (1 - cu)
            b = border > 0
            nb = int(b.sum())
            if nb == 0 or int((b & subj_fg).sum()) / nb < surround_frac:
                continue                                  # not surrounded -> open background
            if float(bp_bg[comp].mean()) > bg_maxprob:
                continue                                  # looks like background -> skip
            add[comp] = 255
    return cv2.bitwise_or(subject_mask, add)


def _set_detect_threshold(state, thresh):
    """Best-effort lower of SAM 3's concept-detector confidence gate so borderline
    subjects aren't dropped. No-ops (with a one-time note) if the build lacks the API."""
    if thresh is None:
        return state
    try:
        ns = sam3_predictor.set_confidence_threshold(thresh, state)
        return ns if ns is not None else state
    except Exception as e:
        if not getattr(_set_detect_threshold, '_warned', False):
            print(f'  [WARN] set_confidence_threshold unavailable ({e}); using model default gate')
            _set_detect_threshold._warned = True
        return state


def _concept_text_persons(state, mask_threshold: float, prompt: str | None = None) -> list:
    """Exhaustive SAM 3 text detection of the concept; returns [(mask_u8, bbox), ...].

    Text-prompt detection is SAM 3's best-tuned path, so it recovers clean person
    silhouettes the bare box-geometric prompt can miss. Each instance carries its bbox
    so it can be matched to a YOLO box."""
    prompt = prompt or CONCEPT_TEXT_PROMPT
    sam3_predictor.reset_all_prompts(state)
    out = sam3_predictor.set_text_prompt(prompt=prompt, state=state)
    ml = out.get('masks_logits') if isinstance(out, dict) else None
    instances = []
    if ml is None or ml.numel() == 0:
        return instances
    ml = ml.float()
    for i in range(len(ml)):
        m = ((ml[i, 0] > mask_threshold).cpu().numpy().astype(np.uint8)) * 255
        ys, xs = np.where(m > 0)
        if xs.size == 0:
            continue
        instances.append((m, (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))))
    return instances


def _bbox_iou(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return float(inter / union) if union > 0 else 0.0


def _match_concept_to_box(instances: list, box, min_iou: float):
    """Pick the text-detected concept instance whose bbox best overlaps `box`."""
    best, best_iou = None, float(min_iou)
    for m, ibox in instances:
        iou = _bbox_iou(ibox, box)
        if iou >= best_iou:
            best, best_iou = m, iou
    return best


def get_foreground_mask(frame_bgr: np.ndarray, yolo_conf: float = 0.3,
                        box_score_ratio: float = 0.3,
                        mask_threshold: float = 0.5,
                        timer: 'StepTimer | None' = None) -> tuple[np.ndarray | None, list]:
    """First-frame seed mask, fusing a forceful segmenter with a topology reference.

    Per YOLO person box:
      (1) CONCEPT topology — a clean person silhouette from SAM 3's concept detector,
          taken from BOTH a box-geometric prompt AND an exhaustive text ("person")
          detection matched to the box by IoU (CONCEPT_TEXT_DETECT). The concept head is
          confidence-gated, so SAM_DETECT_THRESHOLD is lowered first to stop it dropping
          hard subjects (back turned, low contrast) YOLO did detect.
      (2) FORCEFUL interactive — sam3_model.predict_inst(box): box in, mask out, no
          concept gate, so it always fires and recovers held props (and a subject the
          concept head missed). Its additions are then GATED (INTERACTIVE_GATE_BG):
          area the concept silhouette wraps around (e.g. the arm-on-hip <-> torso gap) is
          dropped as background, while outward protrusions (props / missed limbs) and
          YOLO prop points are kept. The gate is skipped when the concept head found
          nothing for a box, so a fully-missed subject is still recovered in full.

    Enclosed holes are then size-limited-filled (MAX_HOLE_FRAC): speckle is sealed, real
    negative space stays open. Returns (mask_u8 | None, person_boxes).
    """
    h, w = frame_bgr.shape[:2]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # ── YOLO: one pass over ALL classes; split people from holdable props ──────
    with _track(timer, 'yolo'):
        yolo_results = yolo_model(frame_rgb, conf=yolo_conf, verbose=False)[0]

    person_boxes: list = []
    prop_points: list = []                      # (x, y) pixel centres of held props
    if yolo_results.boxes is not None and len(yolo_results.boxes) > 0:
        raw_boxes = yolo_results.boxes.xyxy.cpu().numpy()
        confs     = yolo_results.boxes.conf.cpu().numpy()
        clses     = yolo_results.boxes.cls.cpu().numpy().astype(int)

        scored = []
        cx_frame, cy_frame = w / 2, h / 2
        for box, conf, cls in zip(raw_boxes, confs, clses):
            x1, y1, x2, y2 = box
            if cls == 0:                        # person -> scored candidate box
                bw, bh = x2 - x1, y2 - y1
                cx_box, cy_box = (x1 + x2) / 2, (y1 + y2) / 2
                centrality = 1.0 - (abs(cx_box - cx_frame) / cx_frame * 0.5 +
                                    abs(cy_box - cy_frame) / cy_frame * 0.5)
                area_frac = (bw * bh) / (w * h)
                score = float(conf) * (1 + centrality) * (1 + area_frac * 3)
                scored.append((score, box))
            elif (MASK_HELD_OBJECTS and cls in HELD_OBJECT_CLASSES and
                  (x2 - x1) * (y2 - y1) <= HELD_OBJECT_MAX_FRAC * w * h):
                prop_points.append(((x1 + x2) / 2, (y1 + y2) / 2))   # held-prop point

        if scored:
            best_score   = max(s for s, _ in scored)
            person_boxes = [b for s, b in scored if s >= box_score_ratio * best_score]

    # ── SAM 3 — concept topology (clean) fused with forceful interactive head ──
    with _track(timer, 'sam'), torch.no_grad(), \
         torch.autocast(device_type='cuda', dtype=torch.bfloat16):

        if person_boxes:
            state = sam3_predictor.set_image(Image.fromarray(frame_rgb))
            state = _set_detect_threshold(state, SAM_DETECT_THRESHOLD)

            # exhaustive text detection once per frame; matched to each box below
            concept_instances = (_concept_text_persons(state, mask_threshold)
                                 if CONCEPT_TEXT_DETECT else [])

            combined_mask = np.zeros((h, w), dtype=np.uint8)
            dropped_bg = np.zeros((h, w), dtype=np.uint8)   # background the gate removed
            for box in person_boxes:
                x1, y1, x2, y2 = (float(v) for v in box)

                # (1) CONCEPT topology — box-geometric prompt UNION matched text instance
                concept_box = np.zeros((h, w), dtype=np.uint8)
                box_norm = [((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h,
                            (x2 - x1) / w, (y2 - y1) / h]
                sam3_predictor.reset_all_prompts(state)
                out = sam3_predictor.add_geometric_prompt(box=box_norm, label=True,
                                                          state=state)
                ml = out['masks_logits']
                if ml is not None and ml.numel() > 0:
                    sc = out['scores'].float()
                    concept_box = cv2.bitwise_or(concept_box,
                        ((ml[int(torch.argmax(sc)), 0].float() > mask_threshold)
                         .cpu().numpy().astype(np.uint8)) * 255)
                matched = _match_concept_to_box(concept_instances,
                                                (x1, y1, x2, y2), CONCEPT_MATCH_IOU)
                if matched is not None:
                    concept_box = cv2.bitwise_or(concept_box, matched)
                combined_mask = cv2.bitwise_or(combined_mask, concept_box)

                # (2) FORCEFUL interactive head — always fires; recovers props and a
                #     subject the concept head missed, then gated against concept topology.
                if FORCE_INTERACTIVE:
                    pts = [(px, py) for (px, py) in prop_points
                           if x1 <= px <= x2 and y1 <= py <= y2]
                    pc = np.array(pts, dtype=np.float32) if pts else None
                    pl = np.ones(len(pts), dtype=np.int32) if pts else None
                    masks, ious, _low = sam3_model.predict_inst(
                        state, point_coords=pc, point_labels=pl,
                        box=np.array([x1, y1, x2, y2], dtype=np.float32),
                        multimask_output=True, return_logits=True)
                    forced = (masks[int(np.argmax(ious))] > INTERACTIVE_MASK_THRESHOLD
                              ).astype(np.uint8) * 255
                    forced = _keep_components_in_box(forced, x1, y1, x2, y2,
                                                     BOX_CLIP_MARGIN)
                    # drop background the concept silhouette wraps around; keep outward
                    # props / limbs. Skipped if concept found nothing for this box.
                    if INTERACTIVE_GATE_BG and concept_box.any():
                        gated = _drop_background_regions(forced, concept_box, frame_rgb,
                                                         prop_pts=pts, margin=GATE_BG_MARGIN)
                        dropped_bg = cv2.bitwise_or(
                            dropped_bg, cv2.bitwise_and(forced, cv2.bitwise_not(gated)))
                        forced = gated
                    combined_mask = cv2.bitwise_or(combined_mask, forced)

            if RECOVER_HELD_OBJECTS:
                combined_mask = _recover_held_objects(combined_mask, frame_rgb, person_boxes,
                                                      surround_frac=RECOVER_SURROUND_FRAC,
                                                      bg_maxprob=RECOVER_BG_MAXPROB)
            if FILL_MASK_HOLES:
                combined_mask = _fill_holes(combined_mask, MAX_HOLE_FRAC)
            # the appearance gate is authoritative: never let recovery / hole-fill re-add
            # the background it deliberately dropped (e.g. the arm-on-hip <-> torso gap)
            if dropped_bg.any():
                combined_mask = cv2.bitwise_and(combined_mask, cv2.bitwise_not(dropped_bg))
            return (combined_mask if combined_mask.any() else None), person_boxes

        # ── Fallback: YOLO found no person -> SAM 3 text-prompt concept path ────
        print('  [WARN] No person detected by YOLO — using SAM 3 text-prompt fallback')
        state = sam3_predictor.set_image(Image.fromarray(frame_rgb))
        state = _set_detect_threshold(state, SAM_DETECT_THRESHOLD)
        sam3_predictor.reset_all_prompts(state)
        out = sam3_predictor.set_text_prompt(prompt=CONCEPT_TEXT_PROMPT, state=state)
        masks_logits = out['masks_logits']
        if masks_logits is None or masks_logits.numel() == 0:
            return None, []
        masks_logits = masks_logits.float()
        combined_mask = np.zeros((h, w), dtype=np.uint8)
        for mi in range(len(masks_logits)):
            m = ((masks_logits[mi, 0] > mask_threshold)
                 .cpu().numpy().astype(np.uint8)) * 255
            combined_mask = cv2.bitwise_or(combined_mask, m)
        if FILL_MASK_HOLES:
            combined_mask = _fill_holes(combined_mask, MAX_HOLE_FRAC)
        return (combined_mask if combined_mask.any() else None), []

# ─────────────────────────────────────────────────────────────────────────────
#  Small reusable helpers
# ─────────────────────────────────────────────────────────────────────────────
def make_seed_overlay(frame_bgr: np.ndarray, mask: np.ndarray, alpha: float = 0.45,
                      boxes: list | None = None, box_color=(255, 220, 0),
                      box_thickness: int = 2) -> np.ndarray:
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    m   = mask > 127
    rgb[m] = rgb[m] * (1 - alpha) + np.array([255, 40, 40], np.float32) * alpha
    out = np.clip(rgb, 0, 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, (255, 255, 255), 2)
    if boxes:
        for box in boxes:
            x1, y1, x2, y2 = (int(round(v)) for v in box)
            cv2.rectangle(out, (x1, y1), (x2, y2), box_color, box_thickness)
    return out


def composite_result(pha_u8: np.ndarray, fgr_u8: np.ndarray,
                     mode: str, green: np.ndarray) -> np.ndarray:
    pha = pha_u8 / 255.
    fgr = fgr_u8 / 255.
    if mode == 'greenscreen':
        comp = fgr + green * (1.0 - pha)
    elif mode == 'alpha':
        comp = np.repeat(pha, 3, axis=2)
    else:
        comp = fgr
    return np.round(np.clip(comp * 255, 0, 255)).astype(np.uint8)


def thumbnail(img: np.ndarray, max_w: int = 640) -> np.ndarray:
    h, w = img.shape[:2]
    if w <= max_w:
        return img
    return cv2.resize(img, (max_w, int(h * max_w / w)), interpolation=cv2.INTER_AREA)

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 4 — MATANYONE 2 SHOT PROCESSOR
# ─────────────────────────────────────────────────────────────────────────────
def run_matanyone2_on_shot(
    frames_bgr: list[np.ndarray],
    first_frame_mask: np.ndarray,
    n_warmup_static: int = 20,
    n_warmup_motion: int = 20,
    r_erode: int = 0,
    r_dilate: int = 20,
    timer: 'StepTimer | None' = None,
) -> list[dict]:
    with _track(timer, 'matting_pre'):
        processor = InferenceCore(matanyone2_model, cfg=matanyone2_model.cfg)

        vframes = torch.stack([
            torch.from_numpy(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)
            for f in frames_bgr
        ])  # uint8 [T, C, H, W]

        h_orig, w_orig = vframes.shape[-2:]
        resized = False
        if MAX_SIZE > 0 and min(h_orig, w_orig) > MAX_SIZE:
            scale = MAX_SIZE / min(h_orig, w_orig)
            new_h, new_w = int(h_orig * scale), int(w_orig * scale)
            vframes = (F.interpolate(vframes.float(), size=(new_h, new_w), mode='area')
                       .round().clamp_(0, 255).to(torch.uint8))
            resized = True
        h, w = vframes.shape[-2:]

        warmup_static = vframes[0:1].repeat(n_warmup_static, 1, 1, 1)
        n_motion_actual = min(n_warmup_motion, len(frames_bgr))
        warmup_motion = vframes[:n_motion_actual]

        vframes_full = torch.cat([warmup_static, warmup_motion, vframes], dim=0)
        total_warmup = n_warmup_static + n_motion_actual
        total_len    = len(vframes_full)

        mask_np = first_frame_mask.copy()
        if r_dilate > 0:
            mask_np = gen_dilate(mask_np, r_dilate, r_dilate)
        if r_erode > 0:
            mask_np = gen_erosion(mask_np, r_erode, r_erode)

        mask_t = torch.from_numpy(mask_np).float().to(device)
        if resized:
            mask_t = F.interpolate(mask_t.unsqueeze(0).unsqueeze(0),
                                   size=(h, w), mode='nearest')[0, 0]

    objects = [1]
    results = []

    with torch.inference_mode():
        for ti in range(total_len):
            is_static_warmup = ti < n_warmup_static
            is_warmup        = ti < total_warmup
            step_name = 'matting_warmup' if is_warmup else 'matting_infer'

            with _track(timer, step_name):
                image = (vframes_full[ti].float() / 255.).to(device)

                if ti == 0:
                    out_prob = processor.step(image, mask_t, objects=objects)
                    out_prob = processor.step(image, first_frame_pred=True)
                elif is_static_warmup:
                    out_prob = processor.step(image, first_frame_pred=True)
                else:
                    out_prob = processor.step(image)

                alpha_t = processor.output_prob_to_mask(out_prob)

            if not is_warmup:
                with _track(timer, 'matting_post'):
                    image_np = vframes_full[ti].permute(1, 2, 0).numpy()
                    pha_np   = alpha_t.unsqueeze(2).cpu().float().numpy()
                    fgr_np   = (image_np / 255.) * pha_np

                    pha_out = np.round(np.clip(pha_np * 255, 0, 255)).astype(np.uint8)
                    fgr_out = np.round(np.clip(fgr_np * 255, 0, 255)).astype(np.uint8)

                    if resized:
                        pha_out = cv2.resize(pha_out[..., 0], (w_orig, h_orig),
                                             interpolation=cv2.INTER_LINEAR)[..., None]
                        fgr_out = cv2.resize(fgr_out, (w_orig, h_orig),
                                             interpolation=cv2.INTER_LINEAR)

                    results.append({'pha': pha_out, 'fgr': fgr_out})

    return results


print('✅ Pipeline functions defined')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  FULL PIPELINE  (batch over every video in INPUT_DIR)
# ─────────────────────────────────────────────────────────────────────────────
import gc, glob

all_videos = []
for ext in VIDEO_EXTENSIONS:
    all_videos.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
all_videos = sorted(all_videos)
print(f'Found {len(all_videos)} video(s) to process')

grand_timer   = StepTimer()
grand_frames  = 0
preview_shots = []
last_results  = []

for video_idx, INPUT_VIDEO in enumerate(all_videos):
    base_name   = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
    output_path = os.path.join(OUTPUT_DIR, f'{base_name}_matte.mp4')
    alpha_path  = os.path.join(OUTPUT_DIR, f'{base_name}_alpha.mp4')
    seed_dir    = os.path.join(OUTPUT_DIR, f'{base_name}_seed_masks')

    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        print(f'\n[{video_idx+1}/{len(all_videos)}] Skipping (already done): {base_name}')
        continue

    print(f'\n{"="*70}\n[{video_idx+1}/{len(all_videos)}] Processing: {base_name}\n{"="*70}')
    timer = StepTimer()
    preview_shots = []

    try:
        cap = cv2.VideoCapture(INPUT_VIDEO)
        fps    = cap.get(cv2.CAP_PROP_FPS)
        total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        print(f'Video: {total} frames @ {fps:.2f}fps  |  {width}×{height}')

        print('\n[1/5] Decoding frames ...')
        with timer.track('decode'):
            cap = cv2.VideoCapture(INPUT_VIDEO)
            all_frames = []
            pbar = tqdm(total=total if total > 0 else None, desc='Decoding')
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                all_frames.append(frame)
                pbar.update(1)
            pbar.close()
            cap.release()
        total = len(all_frames)
        print(f'Decoded {total} frames')

        print('\n[2/5] Detecting scene cuts ...')
        with timer.track('scene_cut'):
            cut_starts, _ = compute_scene_cuts(all_frames, threshold=CUT_THRESHOLD)
        shot_intervals = [
            (s, cut_starts[i + 1] if i + 1 < len(cut_starts) else total)
            for i, s in enumerate(cut_starts)
        ]
        print(f'Found {len(shot_intervals)} shot(s): {shot_intervals}')

        print('\n[3/5] Segmenting + matting shots ...')
        if SAVE_SEED_MASKS:
            os.makedirs(seed_dir, exist_ok=True)
        all_results = []

        for shot_idx, (shot_start, shot_end) in enumerate(shot_intervals):
            shot_frames = all_frames[shot_start:shot_end]
            n_frames    = len(shot_frames)
            print(f'\n  Shot {shot_idx+1}/{len(shot_intervals)}: '
                  f'frames {shot_start}–{shot_end-1} ({n_frames} frames)')
            if n_frames == 0:
                continue

            first_frame_mask, yolo_boxes = get_foreground_mask(
                shot_frames[0], yolo_conf=YOLO_CONF,
                box_score_ratio=BOX_SCORE_RATIO,
                mask_threshold=SAM_MASK_THRESHOLD,
                timer=timer)

            if first_frame_mask is None:
                print(f'  [WARN] No foreground found — writing blank alpha for shot {shot_idx+1}.')
                h, w = shot_frames[0].shape[:2]
                all_results.extend({'pha': np.zeros((h, w, 1), np.uint8),
                                    'fgr': np.zeros((h, w, 3), np.uint8)} for _ in shot_frames)
                continue

            with timer.track('mask_debug'):
                overlay = make_seed_overlay(shot_frames[0], first_frame_mask, boxes=yolo_boxes)
                if SAVE_SEED_MASKS:
                    cv2.imwrite(os.path.join(seed_dir, f'shot_{shot_idx+1:02d}_mask.png'),
                                first_frame_mask)
                    cv2.imwrite(os.path.join(seed_dir, f'shot_{shot_idx+1:02d}_overlay.png'),
                                cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

            shot_results = run_matanyone2_on_shot(
                shot_frames, first_frame_mask,
                n_warmup_static=N_WARMUP_STATIC, n_warmup_motion=N_WARMUP_MOTION,
                r_erode=R_ERODE, r_dilate=R_DILATE, timer=timer)
            all_results.extend(shot_results)

            matted0 = composite_result(shot_results[0]['pha'], shot_results[0]['fgr'],
                                       'greenscreen', GREEN)
            preview_shots.append({'start': shot_start, 'end': shot_end,
                                  'overlay': thumbnail(overlay), 'matted': thumbnail(matted0),
                                  'n_boxes': len(yolo_boxes)})

            torch.cuda.empty_cache(); gc.collect()

        print(f'\nProcessed {len(all_results)} output frames total')

        print('\n[4/5] Compositing ...')
        composite_frames, alpha_frames = [], []
        with timer.track('composite'):
            for r in tqdm(all_results, desc='Compositing'):
                composite_frames.append(composite_result(r['pha'], r['fgr'], OUTPUT_MODE, GREEN))
                alpha_frames.append(r['pha'])

        print('\n[5/5] Encoding ...')
        enc = dict(codec='libx264', fps=fps,
                   output_params=['-preset', ENCODER_PRESET, '-crf', str(ENCODER_CRF)])
        tmp_output, tmp_alpha = output_path + '.tmp.mp4', alpha_path + '.tmp.mp4'
        with timer.track('encode'):
            imageio.mimwrite(tmp_output, composite_frames, **enc)
            os.rename(tmp_output, output_path)

            if WRITE_ALPHA_VIDEO:
                imageio.mimwrite(tmp_alpha, [a[..., 0] for a in alpha_frames], **enc)
                os.rename(tmp_alpha, alpha_path)

            if OUTPUT_MODE == 'transparent':
                rgba_dir = os.path.join(OUTPUT_DIR, f'{base_name}_rgba_frames')
                os.makedirs(rgba_dir, exist_ok=True)
                for fi, (rgb, a) in enumerate(zip(composite_frames, alpha_frames)):
                    Image.fromarray(np.concatenate([rgb, a], axis=2)).save(
                        os.path.join(rgba_dir, f'{fi:05d}.png'))
                print(f'RGBA frames saved to {rgba_dir}')

        print(f'\n✅ Done: {base_name}')
        print(f'   Composite : {output_path}')
        if WRITE_ALPHA_VIDEO:
            print(f'   Alpha     : {alpha_path}')
        if SAVE_SEED_MASKS:
            print(f'   Seed masks: {seed_dir}')

        n_out = len(all_results)
        timer.report(title=base_name, n_frames=n_out)
        grand_timer.merge(timer); grand_frames += n_out

        last_results = all_results
        last_shot_intervals = shot_intervals

        del all_frames, composite_frames, alpha_frames
        torch.cuda.empty_cache(); gc.collect()

    except Exception as e:
        print(f'\n❌ FAILED: {base_name} — {e}\n   Continuing to next video ...')
        for tmp in [output_path + '.tmp.mp4', alpha_path + '.tmp.mp4']:
            if os.path.exists(tmp):
                os.remove(tmp)
        torch.cuda.empty_cache(); gc.collect()
        continue

print(f'\n{"="*70}\nAll videos processed.')

if grand_timer.totals:
    grand_timer.report(title='ALL VIDEOS — grand total', n_frames=grand_frames)

In [ ]:
import matplotlib.pyplot as plt

if not preview_shots:
    print('No shots to preview (run the pipeline cell first).')
else:
    n = len(preview_shots)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.2 * n))
    axes = np.array(axes).reshape(n, 2)
    for i, sh in enumerate(preview_shots):
        axes[i, 0].imshow(sh['overlay'])
        axes[i, 0].set_title(
            f"Shot {i+1} (frames {sh['start']}–{sh['end']-1})  ·  "
            f"{sh.get('n_boxes', 0)} YOLO box(es) + SAM 3 mask",
            fontsize=9)
        axes[i, 1].imshow(sh['matted'])
        axes[i, 1].set_title(f"Shot {i+1}  ·  after MatAnyone 2 matting", fontsize=9)
        axes[i, 0].axis('off'); axes[i, 1].axis('off')

    plt.suptitle('Left: YOLO boxes (yellow) + SAM 3 seed mask (red/white outline)   |   Right: final matte',
                 fontsize=12)
    plt.tight_layout()
    out_png = os.path.join(OUTPUT_DIR, 'shot_preview.png')
    plt.savefig(out_png, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Shot preview saved -> {out_png}')

In [ ]:
from google.colab import files

print(f'Downloading: {output_path}')
files.download(output_path)

In [ ]:
import matplotlib.pyplot as plt

# ── Pick which video to inspect ───────────────────────────────────────────────
# Defaults to the first video found in INPUT_DIR. Change the index or set an
# explicit path (e.g. INSPECT_VIDEO = '/content/videos/mv.mp4') to inspect another.
INSPECT_VIDEO = all_videos[0] if all_videos else INPUT_VIDEO
print(f'Inspecting: {INSPECT_VIDEO}')

cap = cv2.VideoCapture(INSPECT_VIDEO)
frames_for_plot = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames_for_plot.append(frame)
cap.release()
print(f'Decoded {len(frames_for_plot)} frames')

cut_starts, scores = compute_scene_cuts(frames_for_plot, threshold=CUT_THRESHOLD)

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(scores, linewidth=0.8, color='steelblue', label='HSV histogram diff')
ax.axhline(CUT_THRESHOLD, color='crimson', linestyle='--', linewidth=1.2,
           label=f'Current threshold ({CUT_THRESHOLD})')
for c in cut_starts[1:]:   # skip the implicit 0-boundary
    ax.axvline(c, color='orange', linewidth=0.6, alpha=0.6)
ax.set_xlabel('Frame')
ax.set_ylabel('Bhattacharyya distance')
ax.set_title(f'Scene cut signal — {len(cut_starts)} shot(s) detected   |   {os.path.basename(INSPECT_VIDEO)}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Cut boundaries at threshold {CUT_THRESHOLD}: {cut_starts}')


In [ ]:
# ──── DIAGNOSTIC · seed-mask stages (concept / interactive / gate / final) ────
# Runs the REAL seed logic on the first frame so you can see exactly where a held prop
# or a body gap ends up, and WHICH stage is responsible:
#   1 concept silhouette        2 interactive RAW  (is the prop even captured here?)
#   3 gate decision  (red = kept extra, blue = dropped as background)
#   4 FINAL seed                5 seed + dilate (what matting receives)
# Per interactive-added region it also prints bg-likeness vs subject-likeness so you can
# tune GATE_BG_MARGIN. Always runs both heads, ignoring FORCE_INTERACTIVE.
import glob
import matplotlib.pyplot as plt

_vids = sorted(sum([glob.glob(os.path.join(INPUT_DIR, e)) for e in VIDEO_EXTENSIONS], []))
assert _vids, f'no videos in {INPUT_DIR}'
DIAG_VIDEO = _vids[5]
print('Diagnosing first frame of:', os.path.basename(DIAG_VIDEO))

cap = cv2.VideoCapture(DIAG_VIDEO); ok, frame_bgr = cap.read(); cap.release()
assert ok, 'could not read first frame'
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
h, w = frame_bgr.shape[:2]

res = yolo_model(frame_rgb, conf=YOLO_CONF, verbose=False)[0]
rb = res.boxes.xyxy.cpu().numpy(); cf = res.boxes.conf.cpu().numpy()
cl = res.boxes.cls.cpu().numpy().astype(int)
scored, prop_points = [], []
for b, c, k in zip(rb, cf, cl):
    x1, y1, x2, y2 = b
    if k == 0:
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        centrality = 1.0 - (abs(cx - w/2)/(w/2)*0.5 + abs(cy - h/2)/(h/2)*0.5)
        area_frac = ((x2 - x1)*(y2 - y1))/(w*h)
        scored.append((float(c)*(1 + centrality)*(1 + area_frac*3), b))
    elif (MASK_HELD_OBJECTS and k in HELD_OBJECT_CLASSES and
          (x2 - x1)*(y2 - y1) <= HELD_OBJECT_MAX_FRAC*w*h):
        prop_points.append(((x1 + x2)/2, (y1 + y2)/2))
assert scored, 'no person detected by YOLO on the first frame'
bs = max(s for s, _ in scored)
person_boxes = [b for s, b in scored if s >= BOX_SCORE_RATIO*bs]
print(f'{len(person_boxes)} person box(es), {len(prop_points)} YOLO prop point(s)')

concept_u = np.zeros((h, w), np.uint8)
inter_u   = np.zeros((h, w), np.uint8)   # interactive RAW (before gate)
gated_u   = np.zeros((h, w), np.uint8)   # interactive after gate
dropped_u = np.zeros((h, w), np.uint8)   # what the gate removed

with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    state = sam3_predictor.set_image(Image.fromarray(frame_rgb))
    state = _set_detect_threshold(state, SAM_DETECT_THRESHOLD)
    concept_instances = _concept_text_persons(state, SAM_MASK_THRESHOLD) if CONCEPT_TEXT_DETECT else []
    print(f'text detection found {len(concept_instances)} person instance(s)')

    for box in person_boxes:
        x1, y1, x2, y2 = (float(v) for v in box)
        concept_box = np.zeros((h, w), np.uint8)
        box_norm = [((x1 + x2)/2)/w, ((y1 + y2)/2)/h, (x2 - x1)/w, (y2 - y1)/h]
        sam3_predictor.reset_all_prompts(state)
        out = sam3_predictor.add_geometric_prompt(box=box_norm, label=True, state=state)
        ml = out['masks_logits']
        if ml is not None and ml.numel() > 0:
            sc = out['scores'].float()
            concept_box = cv2.bitwise_or(concept_box,
                ((ml[int(torch.argmax(sc)), 0].float() > SAM_MASK_THRESHOLD)
                 .cpu().numpy().astype(np.uint8)) * 255)
        matched = _match_concept_to_box(concept_instances, (x1, y1, x2, y2), CONCEPT_MATCH_IOU)
        if matched is not None:
            concept_box = cv2.bitwise_or(concept_box, matched)
        concept_u = cv2.bitwise_or(concept_u, concept_box)

        pts = [(px, py) for (px, py) in prop_points if x1 <= px <= x2 and y1 <= py <= y2]
        pc = np.array(pts, np.float32) if pts else None
        pl = np.ones(len(pts), np.int32) if pts else None
        masks, ious, _ = sam3_model.predict_inst(state, point_coords=pc, point_labels=pl,
            box=np.array([x1, y1, x2, y2], np.float32), multimask_output=True, return_logits=True)
        forced = (masks[int(np.argmax(ious))] > INTERACTIVE_MASK_THRESHOLD).astype(np.uint8) * 255
        forced = _keep_components_in_box(forced, x1, y1, x2, y2, BOX_CLIP_MARGIN)
        inter_u = cv2.bitwise_or(inter_u, forced)

        gated = (_drop_background_regions(forced, concept_box, frame_rgb, prop_pts=pts, margin=GATE_BG_MARGIN)
                 if (INTERACTIVE_GATE_BG and concept_box.any()) else forced)
        gated_u   = cv2.bitwise_or(gated_u, gated)
        dropped_u = cv2.bitwise_or(dropped_u, cv2.bitwise_and(forced, cv2.bitwise_not(gated)))

        # why each interactive-added region was kept/dropped
        extra = cv2.bitwise_and(forced, cv2.bitwise_not(concept_box))
        if concept_box.any() and extra.any():
            hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
            subj_est = cv2.dilate(cv2.bitwise_or(forced, concept_box),
                                  cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25)))
            bgs = (subj_est == 0).astype(np.uint8); sjs = (concept_box > 0).astype(np.uint8)
            if int(bgs.sum()) >= 100:
                hb = cv2.calcHist([hsv], [0, 1], bgs, [36, 50], [0, 180, 0, 256]); cv2.normalize(hb, hb, 0, 255, cv2.NORM_MINMAX)
                hs = cv2.calcHist([hsv], [0, 1], sjs, [36, 50], [0, 180, 0, 256]); cv2.normalize(hs, hs, 0, 255, cv2.NORM_MINMAX)
                pbg = cv2.calcBackProject([hsv], [0, 1], hb, [0, 180, 0, 256], 1).astype(np.float32)
                psj = cv2.calcBackProject([hsv], [0, 1], hs, [0, 180, 0, 256], 1).astype(np.float32)
                nn, lab, st, cen = cv2.connectedComponentsWithStats(extra, connectivity=8)
                for L in range(1, nn):
                    a = int(st[L, cv2.CC_STAT_AREA])
                    if a < 64:
                        continue
                    cm = lab == L
                    mb, ms = float(pbg[cm].mean()), float(psj[cm].mean())
                    dec = 'DROP(bg)' if mb > ms + GATE_BG_MARGIN * 255 else 'keep'
                    cx, cy = int(cen[L][0]), int(cen[L][1])
                    print(f'    extra @({cx:>4},{cy:>4}) area={a:>7}  bg={mb:6.1f}  subj={ms:6.1f}  -> {dec}')

final_seed = cv2.bitwise_or(concept_u, gated_u)
if FILL_MASK_HOLES:
    final_seed = _fill_holes(final_seed, MAX_HOLE_FRAC)
try:
    seed_dil = gen_dilate(final_seed.copy(), R_DILATE, R_DILATE) if R_DILATE > 0 else final_seed.copy()
except NameError:
    seed_dil = (cv2.dilate(final_seed, np.ones((R_DILATE, R_DILATE), np.uint8))
                if R_DILATE > 0 else final_seed.copy())

def _ov(mask, color=(255, 40, 40)):
    o = frame_rgb.astype(np.float32).copy(); sel = mask > 127
    o[sel] = o[sel] * 0.5 + np.array(color, np.float32) * 0.5
    return np.clip(o, 0, 255).astype(np.uint8)

dec_img = frame_rgb.astype(np.float32).copy()
kept_extra = cv2.bitwise_and(gated_u, cv2.bitwise_not(concept_u))
dec_img[kept_extra > 127] = dec_img[kept_extra > 127] * 0.4 + np.array([255, 40, 40], np.float32) * 0.6
dec_img[dropped_u > 127]  = dec_img[dropped_u > 127] * 0.4 + np.array([40, 90, 255], np.float32) * 0.6
dec_img = np.clip(dec_img, 0, 255).astype(np.uint8)

panels = [('1. concept silhouette', _ov(concept_u)),
          ('2. interactive RAW (prop captured?)', _ov(inter_u)),
          ('3. gate: red=kept extra, blue=dropped', dec_img),
          ('4. FINAL seed', _ov(final_seed)),
          (f'5. seed + dilate {R_DILATE}px', _ov(seed_dil))]
fig, axs = plt.subplots(1, 5, figsize=(30, 7))
for ax, (t, im) in zip(axs, panels):
    ax.imshow(im); ax.set_title(t, fontsize=9); ax.axis('off')
plt.suptitle('If the prop is missing in panel 2 -> interactive did not capture it. '
             'If it is BLUE in panel 3 -> the gate dropped it.', fontsize=12)
plt.tight_layout()
out_png = os.path.join(OUTPUT_DIR, 'seed_diagnostic.png')
plt.savefig(out_png, dpi=120, bbox_inches='tight'); plt.show()
print('saved ->', out_png)
